# 04 — SFT v2 → đánh giá → DPO → đánh giá (Colab)

Notebook chạy nối tiếp các bước còn lại của Giai đoạn 3–4, theo đúng thứ tự:

| Bước | Việc | Thời gian (ước tính thô, T4 / L4) | Chi phí OpenAI |
|---|---|---|---|
| 0 | Sinh lại Qwen gốc trên dev **có reranker** (mốc so sánh mới) — chạy được ngay | ~50 / ~25 phút | ~0,5 USD |
| 0R | (Tuỳ chọn) Ablation retriever: Qwen gốc + bkai (RAG v2) so với Qwen gốc có reranker — chỉ để báo cáo | ~50 / ~25 phút | ~0,3–0,5 USD |
| 1 | Dữ liệu SFT v2: `retrieve → answer → filter` → **bác sĩ duyệt** → `build` | ~15–30 phút + thời gian bác sĩ duyệt | ~1–2 USD |
| 2 | Train SFT v3 (Qwen3-8B, QLoRA 4-bit, 3 epoch) | ~1–2 giờ / ~40 phút | 0 |
| 3 | Đánh giá SFT v3 trên dev (114 case) + cổng quyết định | ~50 / ~25 phút | ~0,5 USD |
| 3G | Gemma3-4B (ứng viên thứ hai): gốc có reranker → SFT v3.1 → đánh giá dev + cổng → so với Qwen | ~3–4 giờ / ~1,5 giờ | ~1 USD |
| 4 | DPO: sinh 4 ứng viên × 300 câu → chấm → tạo cặp → train | ~4–6 / ~2–3 giờ | ~2–4 USD |
| 5 | Đánh giá DPO trên dev, so sánh Base / SFT / DPO | ~50 / ~25 phút | ~0,5 USD |

**Trước khi chạy**
- Runtime → Change runtime type → GPU. Khuyến nghị **L4 hoặc A100**; T4 chạy được nhưng chậm, DPO có thể phải giảm `max_length` (xem bước 4d).
- Secrets (biểu tượng chìa khóa bên trái, bật *Notebook access*): `OPENAI_API_KEY` (bắt buộc), `HF_TOKEN` (khuyến nghị).
- Code mới (`scripts/eval_adapter.py`, `scripts/dpo_pipeline.py`, ...) phải có trên nhánh GitHub mà biến `BRANCH` trỏ tới.
- Kết quả lưu vào thư mục **KLTN** (Drive → *Được chia sẻ với tôi*), trong thư mục con `gout_llmops_work`.
  Colab chỉ thấy thư mục được chia sẻ khi nó có **lối tắt trong Drive của tôi**: trên Google Drive, chuột phải
  *KLTN* → *Sắp xếp* → *Thêm lối tắt* → *Drive của tôi*. Tài khoản chạy notebook cần quyền **Người chỉnh sửa** trên KLTN.
- Không chạy notebook từ hai tài khoản cùng lúc: cả hai sẽ ghi vào cùng cache và file ứng viên trên KLTN.

**Nếu Colab bị ngắt:** chạy lại các ô 0–3, rồi chạy tiếp từ bước đang dở. Cache của LLM/judge và file ứng viên DPO
nằm thẳng trên Google Drive nên không mất và không tốn tiền lại; các kết quả khác được lưu lên Drive bằng `save()` cuối mỗi bước.

Tập **test** (267 case) không dùng ở notebook này: mọi lựa chọn model/checkpoint chỉ dựa trên **dev**.

## 0. Cấu hình

In [ ]:
REPO_URL = "https://github.com/NtQuang-23521287/Developing_A_Vietnamese_Gout_Chatbot_Based_On_Large_Language_Models.git"
BRANCH = "main"                                         # nhánh chứa code DPO mới
# Nơi lưu kết quả lâu dài: thư mục KLTN (được chia sẻ) / gout_llmops_work. Đường dẫn đầy đủ được tìm ở ô 1.
KLTN_FOLDER_ID = "1ptzef1HYRBwtKzHAFXW1e4cQ_9xhyKb4"   # lấy từ URL drive.google.com/drive/folders/<ID>
DRIVE_SUBDIR = "gout_llmops_work"

# Mốc so sánh = Qwen3-8B gốc sinh lại ở bước 0 CÓ reranker. Run Giai đoạn 2 (OLD_BASE_RUN) chạy KHÔNG có reranker
# (lỗi nạp bị nuốt) trong khi dữ liệu SFT v2 truy xuất CÓ reranker -> không dùng làm mốc cho cổng quyết định.
BASE_NAME = "qwen3_8b_base_rerank"
OLD_BASE_RUN = "artifacts/runs/20261025T074006Z_baseline_qwen3_8b_7963"
# SFT v3.1 = SFT v3 train lại trên dữ liệu SFT v2.1 (đã sửa theo góp ý bác sĩ vòng 1, bước 1c)
SFT_CONFIG = "configs/training/sft_qwen3_8b_v3_1.yaml"
SFT_ADAPTER = "artifacts/runs/sft_qwen3_8b_v3_1"
SFT_NAME = "qwen3_8b_sft_v3_1"
# Run SFT v3 (dữ liệu 16/11) đã sinh sẵn 381 case trên máy thuê: điền đường dẫn (trong artifacts/runs/) để
# chấm trên DEV mà không sinh lại (bước 3, ô đầu). Để trống nếu không có.
SFT_V3_RUN_EXISTING = ""
# Câu hỏi cho DPO: 150 mẫu SFT v2.1 (bác sĩ đã xem ở vòng 1; mẫu có ghi chú đã sửa theo ghi chú)
DPO_PROMPTS = "artifacts/sft_v2_1/rows.jsonl"
DPO_CONFIG = "configs/training/dpo_v1.yaml"
DPO_ADAPTER = "artifacts/runs/dpo_qwen3_8b_v1"
# Bác sĩ duyệt dữ liệu SFT (bước 1b/1c)
REVIEW_SAMPLE = "--per-category 20 --full-categories red_flag"   # "" = gửi bác sĩ TOÀN BỘ mẫu

DPO_MAX_PROMPTS = 300   # số câu hỏi dùng để sinh ứng viên
DPO_K = 4               # số ứng viên mỗi câu (đề cương: K = 3–5)

## 1. GPU, Google Drive, secrets

In [ ]:
import os, subprocess, shutil
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip().splitlines() if shutil.which("nvidia-smi") else []
print("GPU:", gpu[0] if gpu else "KHÔNG CÓ GPU — Runtime > Change runtime type")
GPU_MEM_GB = int(gpu[0].split(",")[-1]) / 1024 if gpu else 0
BATCH_PROMPTS = 4 if GPU_MEM_GB >= 22 else 2   # số câu hỏi sinh song song ở bước 4a

from google.colab import drive, userdata
drive.mount("/content/drive")   # vừa thêm lối tắt KLTN sau khi mount? dùng drive.mount("/content/drive", force_remount=True)

# Thư mục được chia sẻ hiện ở .shortcut-targets-by-id/<ID>/ (tìm theo ID, không phụ thuộc lối tắt đặt ở đâu)
import glob
_cands = (glob.glob(f"/content/drive/.shortcut-targets-by-id/{KLTN_FOLDER_ID}/*")
          + ["/content/drive/MyDrive/KLTN"]
          + glob.glob("/content/drive/.shortcut-targets-by-id/*/KLTN"))
KLTN_DIR = next((p for p in _cands if os.path.isdir(p)), None)
assert KLTN_DIR, ("Không thấy thư mục KLTN. Trên Google Drive: chuột phải KLTN > Sắp xếp > Thêm lối tắt > Drive của tôi, "
                  "rồi chạy lại ô này với drive.mount('/content/drive', force_remount=True)")
DRIVE_DIR = f"{KLTN_DIR}/{DRIVE_SUBDIR}"
try:
    os.makedirs(DRIVE_DIR, exist_ok=True)
    with open(f"{DRIVE_DIR}/.write_test", "w") as f:
        f.write("ok")
    os.remove(f"{DRIVE_DIR}/.write_test")
except OSError as e:
    raise PermissionError(f"Không ghi được vào {DRIVE_DIR}: cần quyền Người chỉnh sửa trên thư mục KLTN") from e
print("DRIVE_DIR =", DRIVE_DIR)

for key in ("OPENAI_API_KEY", "HF_TOKEN"):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print(f"[!] Chưa có secret {key}")
assert os.environ.get("OPENAI_API_KEY"), "Cần OPENAI_API_KEY cho answer/filter/judge"

## 2. Lấy code và cài thư viện

In [ ]:
%cd /content
if not os.path.isdir("/content/repo"):
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 -b {BRANCH} {REPO_URL} repo
else:
    !git -C /content/repo pull --ff-only
%cd /content/repo
!git log --oneline -1

!pip install -q unsloth
!pip install -q faiss-cpu "sentence-transformers>=3.0" "FlagEmbedding>=1.2.11" "openai>=1.50" pyyaml pandas
!pip install -q -e . --no-deps

# Luôn thấy gói gout_llmops (cả trong kernel lẫn các lệnh !python), kể cả khi pip install -e ở trên không thành công
import sys
SRC = "/content/repo/src"
if SRC not in os.environ.get("PYTHONPATH", "").split(":"):
    os.environ["PYTHONPATH"] = ":".join(p for p in (SRC, os.environ.get("PYTHONPATH", "")) if p)
if SRC not in sys.path:
    sys.path.insert(0, SRC)
!python --version
!python -c "import gout_llmops; print('gout_llmops OK:', gout_llmops.__file__)"

import importlib.metadata as md
for p in ("torch", "transformers", "trl", "peft", "bitsandbytes", "unsloth", "FlagEmbedding"):
    try:
        print(f"{p:>14} {md.version(p)}")
    except md.PackageNotFoundError:
        print(f"{p:>14} (chưa cài)")

## 3. Lưu trữ trên Drive + kiểm tra nhanh

- Thư mục có cache/ghi dần (`artifacts/sft_v2`, `artifacts/judge_cache`, `artifacts/dpo_v1`) được **trỏ thẳng lên Drive**.
- Các thư mục kết quả khác được đồng bộ bằng `save()` (gọi cuối mỗi bước) và lấy lại bằng `restore()`.

In [ ]:
import json, shutil

LINKED = ["artifacts/sft_v2", "artifacts/judge_cache", "artifacts/dpo_v1"]
SYNCED = ["data/sft_v2", "data/dpo", "artifacts/runs", "artifacts/reports"]

def link_to_drive(p):
    dst = f"{DRIVE_DIR}/{p}"
    if os.path.islink(p):
        return
    if os.path.isdir(dst):   # đã có từ phiên trước: chỉ bổ sung file Drive chưa có, không ghi đè
        if os.path.isdir(p):
            subprocess.run(["rsync", "-a", "--ignore-existing", f"{p}/", f"{dst}/"], check=True)
    elif os.path.isdir(p):
        shutil.copytree(p, dst)
    else:
        os.makedirs(dst)
    shutil.rmtree(p, ignore_errors=True)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    os.symlink(dst, p)

def save():
    for p in SYNCED:
        if os.path.exists(p):
            os.makedirs(f"{DRIVE_DIR}/{p}", exist_ok=True)
            subprocess.run(["rsync", "-a", f"{p}/", f"{DRIVE_DIR}/{p}/"], check=True)
    print("Đã lưu lên", DRIVE_DIR)

def restore():
    for p in SYNCED:
        if os.path.isdir(f"{DRIVE_DIR}/{p}"):
            os.makedirs(p, exist_ok=True)
            subprocess.run(["rsync", "-a", f"{DRIVE_DIR}/{p}/", f"{p}/"], check=True)

def latest_run(name):
    return open(f"artifacts/runs/LATEST_{name}.txt").read().strip()

for p in LINKED:
    link_to_drive(p)
restore()
!python -m pytest -q tests/test_preference.py tests/test_reward_v2.py tests/test_doctor_review.py

## Bước 0 — Qwen gốc có reranker (mốc so sánh mới; chạy được ngay, không cần chờ bác sĩ)

Các run Giai đoạn 2 trên testset v03 chạy **không có reranker** dù `rag_v1.yaml` bật (retriever cũ nuốt lỗi nạp
FlagReranker). Dữ liệu SFT v2/DPO lại được truy xuất **có reranker** (`build_sft_v2.py retrieve` dừng nếu reranker
không nạp được). Để cổng quyết định ở bước 3/5 so cùng một protocol, sinh lại Qwen gốc trên dev với đúng retriever
hiện tại. Retriever giờ **báo lỗi** nếu reranker không nạp được, và `gate` từ chối so hai run khác protocol truy xuất.

Bảng so sánh cuối ô là ablation có/không reranker trên Qwen gốc — đưa được vào khóa luận.

In [ ]:
!python -c "from FlagEmbedding import FlagReranker; FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True); print('Reranker OK')"
!python scripts/eval_adapter.py generate --name {BASE_NAME} --split dev
BASE_RUN = latest_run(BASE_NAME)
!python scripts/eval_adapter.py score --run {BASE_RUN} --split dev
!python scripts/compare_runs_v2.py {OLD_BASE_RUN} {BASE_RUN} --names Base-NoRerank Base-Rerank --split dev --out artifacts/reports/base_rerank_ablation_dev
save()

## Bước 0R (tuỳ chọn) — Ablation retriever: bkai thay MiniLM + reranker

Benchmark truy xuất trên dev (`artifacts/reports/rag/retrieval_benchmark_dev.md`, 104 câu một lượt): với embedding
**bkai** (tách từ pyvi, không reranker), đoạn bằng chứng nằm trong 3 đoạn đưa vào prompt ở **83,7%** câu; cấu hình
đang dùng (MiniLM + reranker) chỉ **64,4%** (chênh +19,2 điểm %, p = 0,001). Ô này đo xem chênh lệch đó có làm
**câu trả lời** tốt hơn không: sinh lại Qwen3-8B gốc trên dev với `configs/rag/rag_v2_bkai.yaml`, chấm Judge v2,
so ghép cặp với Qwen gốc có reranker của bước 0.

- Kết quả chỉ để ghi vào khóa luận (ablation). **SFT, DPO và cổng quyết định vẫn dùng `rag_v1`**; cổng từ chối so
  run này với run `rag_v1` vì khác protocol truy xuất, còn `compare_runs_v2.py` chỉ cảnh báo.
- Chạy lúc nào cũng được sau bước 0. Khoảng 50 phút trên T4 / 25 phút trên L4, khoảng 0,3–0,5 USD.
- Index bkai đã dựng sẵn trong repo (`artifacts/kb/gout_kb_v1_bkai`), không cần dựng lại.
- Đo trên Gemma thay vì Qwen (chạy sau ô G0): `ABL_NAME = "gemma3_4b_base_bkai"`,
  `ABL_MODEL = "configs/models/gemma3_4b.yaml"`, `ABL_REF = latest_run("gemma3_4b_base_rerank")`.

In [ ]:
# 0R — Ablation retriever trên dev: model gốc + RAG v2 (bkai, không reranker) so với model gốc + rag_v1 (bước 0)
!pip install -q "pyvi==0.1.1"
ABL_RAG = "configs/rag/rag_v2_bkai.yaml"
ABL_NAME = "qwen3_8b_base_bkai"
ABL_MODEL = "configs/models/qwen3_8b.yaml"
ABL_REF = latest_run(BASE_NAME)   # Qwen gốc + rag_v1 có reranker (bước 0)
!python scripts/eval_adapter.py generate --name {ABL_NAME} --model-config {ABL_MODEL} --rag-config {ABL_RAG} --split dev
ABL_RUN = latest_run(ABL_NAME)
!python scripts/eval_adapter.py score --run {ABL_RUN} --split dev
!python scripts/compare_runs_v2.py {ABL_REF} {ABL_RUN} --names RAGv1-MiniLM-Rerank RAGv2-bkai --split dev --out artifacts/reports/rag_ablation_{ABL_NAME}_dev
save()

## Bước 1 — Dữ liệu SFT v2 (RAG-aware) + bác sĩ duyệt

> **Trạng thái 19/11:** 1a, 1b đã chạy xong — **không chạy lại** (sẽ ghi đè `filtered.jsonl`, không còn khớp với file bác sĩ đã xem). Bác sĩ đã xem 151 mẫu và ghi chú trực tiếp vào 19 dòng (không điền cột ĐÁNH GIÁ, nên `review_import` không đọc được file này). Nhóm dùng kết quả này, không duyệt lại: dữ liệu SFT chính thức là **SFT v2.1** ở bước 1c.

Bước `questions` đã chạy và commit (641 câu hỏi, đã loại câu trùng/gần trùng testset v03).

### 1a. Truy xuất → viết câu trả lời → lọc tự động
Truy xuất context bằng **đúng retriever lúc suy luận** → gpt-4.1-mini viết câu trả lời chỉ dựa trên context → lọc bằng luật + LLM.

In [ ]:
!python scripts/build_sft_v2.py retrieve
!python scripts/build_sft_v2.py answer
!python scripts/build_sft_v2.py filter
save()

### 1b. Xuất file Excel cho bác sĩ duyệt

File nằm trên Drive: **KLTN/gout_llmops_work/artifacts/sft_v2/review/sft_v2_review.xlsx**, gồm các sheet:
*Hướng dẫn* (đọc trước), *Duyệt* (bác sĩ chọn **Đạt / Cần sửa / Loại**, viết bản sửa ở cột vàng), *Tài liệu* (đoạn tài liệu chatbot được đọc).
Câu hỏi rủi ro cao (dấu hiệu nguy hiểm, bẫy an toàn) xếp lên đầu.

**Gửi bao nhiêu mẫu** (biến `REVIEW_SAMPLE` ở ô 0; dữ liệu hiện có 557 mẫu):

| `REVIEW_SAMPLE` | Số mẫu | Thời gian bác sĩ |
|---|---|---|
| `"--per-category 20"` | 100 (20 mẫu ngẫu nhiên mỗi loại) | ~3–5 giờ |
| `"--per-category 20 --full-categories red_flag"` (mặc định) | 151 (toàn bộ 71 câu dấu hiệu nguy hiểm + 20 mỗi loại còn lại) | ~5–7,5 giờ |
| `""` | 557 (tất cả) | ~19–28 giờ |

Mẫu được chọn ngẫu nhiên có seed, chia đều theo loại câu hỏi, nên tỉ lệ bác sĩ chấm "Đạt" ước lượng được chất lượng của cả bộ dữ liệu.
Mẫu không gửi bác sĩ không được dùng mặc định; muốn thêm vào train thì dùng `fix_sft_v2_data.py --extra-unreviewed` ở bước 1c.

**Gửi cho bác sĩ:** trên Google Drive, chuột phải file → *Chia sẻ* → nhập email bác sĩ, quyền **Người chỉnh sửa**
(chỉ chia sẻ file này, không chia sẻ cả thư mục KLTN). Bác sĩ mở bằng Google Sheets và sửa trực tiếp, hoặc tải về sửa bằng Excel
rồi gửi lại. Khi nhận lại, chép đè vào đúng đường dẫn trên.

Lệnh dưới **không ghi đè** nếu file đã tồn tại, nên chạy lại không làm mất kết quả bác sĩ đã duyệt
(muốn xuất lại với số mẫu khác khi bác sĩ CHƯA bắt đầu duyệt: thêm `--force`).

In [ ]:
!python scripts/build_sft_v2.py review_export {REVIEW_SAMPLE}

### Kết quả bác sĩ xem (vòng 1) — đã nhận

File bác sĩ trả: `artifacts/sft_v2/review/sft_v2_review_vong1_bac_si.xlsx` (trong repo). Bác sĩ không chọn Đạt/Cần sửa/Loại
mà ghi chú trực tiếp vào 19/151 dòng; dòng không có ghi chú được giữ nguyên. Nhóm dùng kết quả này, không gửi duyệt lại.
Chi tiết các góp ý và cách xử lý: `docs/sft_v2_doctor_review.md`.

### 1c. Dữ liệu SFT v2.1 — dữ liệu SFT chính thức

Ngày 16/11 nhóm đã tích hợp 17/19 ghi chú của bác sĩ vào
`data/sft_v2` (bản dùng cho SFT v3). `scripts/fix_sft_v2_data.py` dựng **SFT v2.1** từ bản đó:
- áp `data/sft_v2/doctor_round1_fixes.json`: 2 câu còn khuyên "không tự ý ngưng" khi colchicine gây tiêu chảy, 3 câu thiếu
  ngoại lệ phản ứng thuốc, 2 ghi chú bác sĩ bị bỏ sót (câu hỏi lao kê bị bỏ), đường link trong lời chatbot;
- kiểm tra mọi mẫu bằng luật rút từ góp ý bác sĩ (`gout_llmops/data/doctor_rules.py`); còn vi phạm thì dừng;
- trả lại nhãn nguồn RAG thật (giống hệt lúc suy luận); kiểm tra ngoài các chỗ sửa không ký tự nào khác bị đổi;
- ghi `data/sft_v2_1/` (train/val/manifest/fix_log) và `artifacts/sft_v2_1/rows.jsonl` (câu hỏi cho DPO).

Dữ liệu chỉ có 150 mẫu, gần một nửa là dấu hiệu nguy hiểm (dev: 7%), trong khi dev 81% là tình huống lâm sàng.
Nếu SFT v3 kém base trên dev ở nhóm tình huống lâm sàng, thử thêm mẫu AI đã lọc nhưng chưa gửi bác sĩ:
`--extra-unreviewed artifacts/sft_v2/filtered.jsonl` (chỉ vào train; mẫu vi phạm luật bác sĩ bị bỏ; manifest ghi rõ).


In [ ]:
!python scripts/fix_sft_v2_data.py
# !python scripts/fix_sft_v2_data.py --extra-unreviewed artifacts/sft_v2/filtered.jsonl
!cat data/sft_v2_1/manifest.json
save()


Đọc thử vài mẫu trước khi train:

In [ ]:
import random
rows = [json.loads(l) for l in open("data/sft_v2_1/train.jsonl")]
print(len(rows), "mẫu train")
for r in random.Random(0).sample(rows, 3):
    print("-" * 100)
    print(f"[{r['category']}] HỎI:", r["messages"][-2]["content"])
    print("ĐÁP:", r["messages"][-1]["content"])

## Bước 2 — Train SFT v3.1

`sft_qwen3_8b_v3_1.yaml`: dữ liệu SFT v2.1 (bước 1c), siêu tham số giữ nguyên như SFT v3 (10 epoch, sft.py giữ checkpoint val loss thấp nhất — ở SFT v3 là epoch 5) để so v3/v3.1 chỉ khác dữ liệu; `lora_dropout = 0.05`
(cần > 0 để dùng MC Dropout trên LoRA ở Giai đoạn 5). Preflight chỉ đếm token, không train.

Checkpoint mỗi epoch được chép lên Drive (`--checkpoint-mirror`). Nếu Colab ngắt giữa chừng: chạy lại các ô mục 0–3 rồi ô train, `--resume` sẽ train tiếp từ epoch đã lưu. Script từ chối train tiếp nếu dữ liệu hoặc config khác lần trước, và từ chối train đè lên checkpoint cũ khi thiếu `--resume`.

In [ ]:
!python scripts/train_sft.py --config {SFT_CONFIG} --preflight-only

In [ ]:
# Muốn kiểm tra nhanh trước (2 bước, không lưu adapter): thay --full-test bằng --smoke-test
# Checkpoint mỗi epoch được chép lên Drive. Colab ngắt giữa chừng: chạy lại các ô mục 0–3 rồi ô này -> train tiếp
!python scripts/train_sft.py --config {SFT_CONFIG} --full-test --resume --checkpoint-mirror "{DRIVE_DIR}/{SFT_ADAPTER}"
save()

## Bước 3 — Đánh giá SFT v3 trên dev + cổng quyết định

Sinh câu trả lời theo **đúng protocol Giai đoạn 2** (RAG v1, system prompt, greedy, 512 token) cho 114 case dev,
chấm Judge v2 + Reward v2, rồi so với Qwen gốc trên cùng case.

Cổng **ĐẠT** khi đồng thời: reward ≥ base − 0,02; tỷ lệ omission≥2 ≤ base + 0,05; safety≥2 ≤ base + 0,02;
không thêm câu safety=3 hay fatal nào. Đây là quy ước của nhóm, cần ghi rõ trong khóa luận.
ĐẠT → DPO dùng SFT v3 làm policy. KHÔNG ĐẠT → DPO thẳng trên Qwen gốc, SFT ghi nhận là kết quả âm của ablation.

In [ ]:
# (Tùy chọn) Run SFT v3 đã sinh sẵn trên máy thuê (381 case, dữ liệu 16/11): CHỈ chấm dev và chạy cổng với Qwen gốc có
# reranker. Không chấm, không đọc phần test (eval_adapter từ chối --split test/all nếu không có --final-test).
if SFT_V3_RUN_EXISTING:
    BASE_RUN = latest_run(BASE_NAME)
    from gout_llmops.evaluation.protocol_check import retrieval_profile
    print("Truy xuất SFT v3:", retrieval_profile(SFT_V3_RUN_EXISTING), "| base:", retrieval_profile(BASE_RUN))
    !python scripts/eval_adapter.py score --run {SFT_V3_RUN_EXISTING} --split dev
    !python scripts/eval_adapter.py gate --base {BASE_RUN} --candidate {SFT_V3_RUN_EXISTING} --split dev --out artifacts/reports/gate_qwen3_8b_sft_v3_dev.json
    save()

In [ ]:
!python scripts/eval_adapter.py generate --name {SFT_NAME} --adapter {SFT_ADAPTER} --split dev
SFT_RUN = latest_run(SFT_NAME)
!python scripts/eval_adapter.py score --run {SFT_RUN} --split dev
save()

In [ ]:
BASE_RUN = latest_run(BASE_NAME)   # Qwen gốc có reranker (bước 0)
!python scripts/compare_runs_v2.py {BASE_RUN} {SFT_RUN} --names Qwen-Base Qwen-SFT --split dev --out artifacts/reports/qwen_base_vs_{SFT_NAME}_dev
!python scripts/eval_adapter.py gate --base {BASE_RUN} --candidate {SFT_RUN} --split dev --out artifacts/reports/gate_{SFT_NAME}_dev.json

gate = json.load(open(f"artifacts/reports/gate_{SFT_NAME}_dev.json"))
POLICY_ADAPTER = SFT_ADAPTER if gate["pass"] else "none"
# Muốn tự quyết thì gán lại ở đây, ví dụ: POLICY_ADAPTER = "none"
print("Policy cho DPO:", POLICY_ADAPTER if POLICY_ADAPTER != "none" else "Qwen3-8B gốc")
save()

## Bước 3G — Gemma3-4B (ứng viên thứ hai)

Trên dev (testset v03, Judge v2), Gemma3-4B gốc đạt 0,733 so với Qwen3-8B gốc 0,663 (p = 0,015), nên Gemma được
giữ làm ứng viên qua SFT. Mô hình cuối chọn theo reward trên dev và cổng quyết định sau SFT; DPO làm trên mô hình
được chọn.

- Mốc so sánh của Gemma là **Gemma gốc có reranker** (ô G0), giống cách làm với Qwen ở bước 0.
- Gemma là model *gated*: tài khoản của secret `HF_TOKEN` phải bấm đồng ý điều khoản tại
  https://huggingface.co/google/gemma-3-4b-it. Thiếu quyền sẽ báo `401 GatedRepoError`.
- Chỉ dùng dev; không chạy split test/all.

In [ ]:
import yaml
from huggingface_hub import hf_hub_download

GEMMA_MODEL_CONFIG = "configs/models/gemma3_4b.yaml"
GEMMA_BASE_NAME = "gemma3_4b_base_rerank"
GEMMA_SFT_CONFIG = "configs/training/sft_gemma3_4b_v3_1.yaml"
GEMMA_SFT_ADAPTER = "artifacts/runs/sft_gemma3_4b_v3_1"
GEMMA_SFT_NAME = "gemma3_4b_sft_v3_1"
# Run Gemma SFT v3 (dữ liệu SFT v2) đã sinh sẵn, nếu có: điền đường dẫn trong artifacts/runs/ để CHỈ chấm dev
GEMMA_V3_RUN_EXISTING = ""

# Kiểm tra quyền tải Gemma trước khi chạy lâu
_gcfg = yaml.safe_load(open(GEMMA_MODEL_CONFIG))
try:
    hf_hub_download(_gcfg["model_id"], "config.json", revision=_gcfg.get("revision"))
    print("Gemma: có quyền tải", _gcfg["model_id"])
except Exception as e:
    raise RuntimeError(f"Không tải được Gemma ({type(e).__name__}). 403 = tài khoản của HF_TOKEN chưa bấm đồng ý "
                       "điều khoản tại huggingface.co/google/gemma-3-4b-it: đồng ý xong chỉ cần chạy lại ô này. "
                       "Nếu đổi giá trị secret HF_TOKEN: chạy lại ô dưới mục '1. GPU, Google Drive, secrets' "
                       "rồi chạy lại ô này.") from e

In [ ]:
# G0 — Gemma gốc có reranker trên dev (mốc so sánh cho Gemma)
!python scripts/eval_adapter.py generate --name {GEMMA_BASE_NAME} --model-config {GEMMA_MODEL_CONFIG} --split dev
GEMMA_BASE_RUN = latest_run(GEMMA_BASE_NAME)
!python scripts/eval_adapter.py score --run {GEMMA_BASE_RUN} --split dev
save()

In [ ]:
# G1 — (Tùy chọn) Run Gemma SFT v3 đã sinh sẵn: CHỈ chấm dev và chạy cổng với Gemma gốc có reranker
if GEMMA_V3_RUN_EXISTING:
    GEMMA_BASE_RUN = latest_run(GEMMA_BASE_NAME)
    from gout_llmops.evaluation.protocol_check import retrieval_profile
    print("Truy xuất Gemma SFT v3:", retrieval_profile(GEMMA_V3_RUN_EXISTING), "| base:", retrieval_profile(GEMMA_BASE_RUN))
    !python scripts/eval_adapter.py score --run {GEMMA_V3_RUN_EXISTING} --split dev
    !python scripts/eval_adapter.py gate --base {GEMMA_BASE_RUN} --candidate {GEMMA_V3_RUN_EXISTING} --split dev --out artifacts/reports/gate_gemma3_4b_sft_v3_dev.json
    save()

In [ ]:
# G2 — Train Gemma SFT v3.1 (cùng siêu tham số Gemma SFT v3, dữ liệu SFT v2.1)
# Checkpoint mỗi epoch được chép lên Drive. Colab ngắt giữa chừng: chạy lại các ô mục 0–3, ô cấu hình Gemma
# (đầu Bước 3G) rồi ô này -> train tiếp từ epoch đã lưu
!python scripts/train_sft.py --config {GEMMA_SFT_CONFIG} --preflight-only
!python scripts/train_sft.py --config {GEMMA_SFT_CONFIG} --full-test --resume --checkpoint-mirror "{DRIVE_DIR}/{GEMMA_SFT_ADAPTER}"
save()

In [ ]:
# G3 — Đánh giá Gemma SFT v3.1 trên dev + cổng quyết định (so với Gemma gốc có reranker)
!python scripts/eval_adapter.py generate --name {GEMMA_SFT_NAME} --adapter {GEMMA_SFT_ADAPTER} --model-config {GEMMA_MODEL_CONFIG} --split dev
GEMMA_SFT_RUN = latest_run(GEMMA_SFT_NAME)
!python scripts/eval_adapter.py score --run {GEMMA_SFT_RUN} --split dev
GEMMA_BASE_RUN = latest_run(GEMMA_BASE_NAME)
!python scripts/compare_runs_v2.py {GEMMA_BASE_RUN} {GEMMA_SFT_RUN} --names Gemma-Base Gemma-SFT --split dev --out artifacts/reports/gemma_base_vs_{GEMMA_SFT_NAME}_dev
!python scripts/eval_adapter.py gate --base {GEMMA_BASE_RUN} --candidate {GEMMA_SFT_RUN} --split dev --out artifacts/reports/gate_{GEMMA_SFT_NAME}_dev.json
save()

In [ ]:
# G4 — Chọn mô hình: so các run đã có trên cùng 114 ca dev, đều có reranker (run nào chưa có thì bỏ qua và báo thiếu)
_all = [("Qwen-Base", BASE_NAME), ("Qwen-SFT", SFT_NAME), ("Gemma-Base", GEMMA_BASE_NAME), ("Gemma-SFT", GEMMA_SFT_NAME)]
_have = [(lbl, n) for lbl, n in _all if os.path.exists(f"artifacts/runs/LATEST_{n}.txt")]
print("Chưa có run:", ", ".join(lbl for lbl, n in _all if (lbl, n) not in _have) or "không")
RUNS_STR = " ".join(latest_run(n) for _, n in _have)
NAMES_STR = " ".join(lbl for lbl, _ in _have)
!python scripts/compare_runs_v2.py {RUNS_STR} --names {NAMES_STR} --split dev --out artifacts/reports/qwen_vs_gemma_dev
for _n in (SFT_NAME, GEMMA_SFT_NAME):
    _p = f"artifacts/reports/gate_{_n}_dev.json"
    print(_n, ("QUA cổng" if json.load(open(_p))["pass"] else "KHÔNG qua cổng") if os.path.exists(_p) else "chưa có cổng")
save()

Chọn mô hình cho DPO: bản SFT **qua cổng** có reward dev cao hơn. Nếu không bản nào qua cổng, DPO chạy trên
mô hình gốc có reward dev cao hơn. Bước 4 bên dưới hiện chạy DPO cho Qwen (`dpo_v1.yaml`, `POLICY_ADAPTER` lấy từ
cổng của Qwen). Nếu chọn Gemma, cần thêm config DPO cho Gemma (`base_model_config: configs/models/gemma3_4b.yaml`).

## Bước 4 — DPO

- **4a. Sinh ứng viên**: policy sinh K câu trả lời (sampling, T = 0,8) cho tối đa 300 câu hỏi của dữ liệu SFT v2
  (giữ hết red_flag, safety_trap, hội thoại nhiều lượt). Ghi từng câu lên Drive → bị ngắt thì chạy lại ô, nó làm tiếp.
- **4b. Chấm**: Judge v2 + Reward v2 (công thức 1–3); câu trả lời mẫu đã lọc làm REFERENCE cho judge.
- **4c. Tạo cặp** theo Preference Mining Rule (`src/gout_llmops/training/preference.py`): chosen = R cao nhất trong các câu
  R ≥ 0,5, không fatal, cả 3 rủi ro ≤ 1; rejected = R thấp nhất; giữ khi chênh ≥ 0,15; câu high/critical mà không có ứng viên
  nào đạt thì lấy câu trả lời mẫu làm chosen.
- **4d. Train**: β = 0,1, lr 1e-5, 2 epoch. Reference = chính policy (adapter SFT nạp 2 lần, một bản đóng băng).

In [ ]:
if "POLICY_ADAPTER" not in globals():   # chạy lại sau khi Colab ngắt
    g = f"artifacts/reports/gate_{SFT_NAME}_dev.json"
    POLICY_ADAPTER = (SFT_ADAPTER if json.load(open(g))["pass"] else "none") if os.path.exists(g) else "none"
print("Policy:", POLICY_ADAPTER)

# 4a — bước lâu nhất
!python scripts/dpo_pipeline.py generate --policy-adapter {POLICY_ADAPTER} --k {DPO_K} --max-prompts {DPO_MAX_PROMPTS} --batch-prompts {BATCH_PROMPTS} --prompts {DPO_PROMPTS}

In [ ]:
# 4b + 4c
!python scripts/dpo_pipeline.py score
!python scripts/dpo_pipeline.py mine
save()

Nếu `mine` báo quá ít cặp (< 150): tăng `DPO_MAX_PROMPTS` (sinh tiếp phần còn thiếu), hoặc chạy lại `mine --min-margin 0.10`.
Đọc thử vài cặp để chắc chosen thật sự tốt hơn rejected:

In [ ]:
import random
pairs = [json.loads(l) for l in open("data/dpo/preference_v1.jsonl")]
for p in random.Random(1).sample(pairs, min(3, len(pairs))):
    print("=" * 100)
    print(f"[{p['category']} | {p['source']}] HỎI:", p["prompt"][-1]["content"])
    print(f"\nCHOSEN (R={p['chosen_reward']}):\n{p['chosen'][:1200]}")
    print(f"\nREJECTED (R={p['rejected_reward']:.3f}):\n{p['rejected'][:1200]}")

In [ ]:
# 4d — smoke test 2 bước trước (kiểm tra VRAM và phiên bản TRL)
!python scripts/dpo_pipeline.py train --config {DPO_CONFIG} --policy-adapter {POLICY_ADAPTER} --smoke-test

# Chỉ khi bị "CUDA out of memory" (thường trên T4): bỏ dấu # ở dòng dưới rồi chạy lại ô này
# !sed -i 's/^max_length: 2048/max_length: 1792/; s/^max_prompt_length: 1536/max_prompt_length: 1280/' {DPO_CONFIG}

In [ ]:
!python scripts/dpo_pipeline.py train --config {DPO_CONFIG} --policy-adapter {POLICY_ADAPTER}
save()

logs = json.load(open(f"{DPO_ADAPTER}/log_history.json"))
keys = ["loss", "rewards/margins", "rewards/accuracies", "rewards/chosen", "rewards/rejected"]
print("step  " + "  ".join(f"{k:>18}" for k in keys))
for l in logs:
    if "loss" in l:
        print(f"{l['step']:>4}  " + "  ".join(f"{l.get(k, float('nan')):>18.4f}" for k in keys))

Kỳ vọng: `rewards/accuracies` tăng dần lên > 0,6 và `rewards/margins` dương, tăng dần. Nếu accuracies quanh 0,5 suốt quá trình thì DPO chưa học được gì (dữ liệu nhiễu hoặc lr quá thấp).

## Bước 5 — Đánh giá DPO trên dev, so sánh Base / SFT / DPO

In [ ]:
!python scripts/eval_adapter.py generate --name qwen3_8b_dpo_v1 --adapter {DPO_ADAPTER} --split dev
DPO_RUN = latest_run("qwen3_8b_dpo_v1")
!python scripts/eval_adapter.py score --run {DPO_RUN} --split dev
save()

In [ ]:
BASE_RUN = latest_run(BASE_NAME)   # Qwen gốc có reranker (bước 0)
runs, names = [BASE_RUN], ["Qwen-Base"]
if POLICY_ADAPTER != "none":
    runs.append(latest_run(SFT_NAME)); names.append("Qwen-SFT")
runs.append(DPO_RUN); names.append("Qwen-DPOv1")
RUNS, NAMES, PREV = " ".join(runs), " ".join(names), runs[-2]

!python scripts/compare_runs_v2.py {RUNS} --names {NAMES} --split dev --out artifacts/reports/base_sft_dpo_dev
!python scripts/eval_adapter.py gate --base {PREV} --candidate {DPO_RUN} --split dev --out artifacts/reports/gate_dpo_dev.json
save()

## Sau khi chạy xong

Mọi kết quả nằm trong `DRIVE_DIR` (Drive: *KLTN/gout_llmops_work*). Mang về repo và commit:
- `data/sft_v2/` và `data/dpo/` (jsonl + `manifest.json`): phiên bản dữ liệu Data v1.1 / preference v1.
- `artifacts/sft_v2/review/sft_v2_review.xlsx`, `reviewed.jsonl`, `review_rejected.jsonl`, `review_manifest.json`: bằng chứng bác sĩ duyệt.
- `artifacts/reports/`: bảng so sánh `.md/.json`, kết quả cổng quyết định.
- Các run đánh giá trong `artifacts/runs/` (predictions, `eval_v2/`), trừ file `.safetensors`.

Adapter (`.safetensors`) giữ trên Drive hoặc đưa lên Git LFS như SFT v1.
Tập **test** chỉ chạy **một lần** cho cấu hình cuối cùng: `eval_adapter.py generate ... --split test --final-test`.